# Phase 9: Cross-Dataset Validation (UNSW-NB15)

Goal:
Evaluate whether models trained on CICIDS2017
can detect attacks in a different dataset without retraining.


In [1]:
import pandas as pd

df_unsw = pd.read_csv(
    "../data/UNSW_NB15/UNSW_NB15_testing-set.csv"
)

print(df_unsw.shape)
df_unsw.head()


(175341, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


Understand Labels (VERY IMPORTANT)

UNSW uses:

label → 0 normal, 1 attack

In [2]:
df_unsw['label'].value_counts()


label
1    119341
0     56000
Name: count, dtype: int64

Basic Cleaning

In [3]:
df_unsw = df_unsw.replace([float('inf'), -float('inf')], pd.NA)
df_unsw = df_unsw.dropna()


Feature Selection (MATCH CICIDS STYLE)

Remove non-numeric & metadata columns:

In [4]:
drop_cols = [
    'id', 'attack_cat', 'label'
]

df_features = df_unsw.drop(columns=drop_cols, errors='ignore')
df_features = df_features.select_dtypes(include=['int64', 'float64'])

y_unsw = df_unsw['label']


Scale Using SAME LOGIC

⚠️ Important
We re-fit scaler here because datasets differ.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_unsw = scaler.fit_transform(df_features)

print(X_unsw.shape)


(175341, 39)


# Load Trained Models (NO RETRAINING)

In [6]:
import joblib
from tensorflow.keras.models import load_model

iforest = joblib.load("../models/isolation_forest.pkl")
autoencoder = load_model("../models/autoencoder.keras")
lstm_autoencoder = load_model("../models/lstm_autoencoder.keras")

e:\3rd_year\6th-sem\await\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [7]:
# Feature Alignment for Cross-Dataset Validation
# CICIDS2017 has 78 features, UNSW-NB15 has 39 features
# Padding UNSW features with zeros to match expected input dimension
import numpy as np
print(f"Original UNSW-NB15 shape: {X_unsw.shape}")
print(f"Expected feature count: 78")
# Pad with zeros to reach 78 features
n_missing_features = 78 - X_unsw.shape[1]
zero_padding = np.zeros((X_unsw.shape[0], n_missing_features))
X_unsw_aligned = np.concatenate([X_unsw, zero_padding], axis=1)
print(f"Aligned UNSW-NB15 shape: {X_unsw_aligned.shape}")
# Replace X_unsw with aligned version for model evaluation
X_unsw = X_unsw_aligned

Original UNSW-NB15 shape: (175341, 39)
Expected feature count: 78
Aligned UNSW-NB15 shape: (175341, 78)


In [8]:
# Isolation Forest Predictions on UNSW-NB15
iforest_scores_unsw = iforest.decision_function(X_unsw)
print(f"Isolation Forest scores shape: {iforest_scores_unsw.shape}")

Isolation Forest scores shape: (175341,)


In [9]:
X_recon = autoencoder.predict(X_unsw)
ae_errors_unsw = ((X_unsw - X_recon) ** 2).mean(axis=1)


5480/5480 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step


In [10]:
np.save("../data/unsw/unsw_ae_errors.npy", ae_errors_unsw)
print("Saved: unsw_ae_errors.npy")


Saved: unsw_ae_errors.npy


In [11]:
import numpy as np

def create_sequences(X, window=10):
    return np.array([X[i:i+window] for i in range(len(X) - window)])

X_seq_unsw = create_sequences(X_unsw)
X_seq_pred = lstm_autoencoder.predict(X_seq_unsw)

lstm_errors_unsw = ((X_seq_unsw - X_seq_pred) ** 2).mean(axis=(1,2))


5480/5480 ━━━━━━━━━━━━━━━━━━━━ 22s 4ms/step


In [12]:
offset = len(X_unsw) - len(lstm_errors_unsw)

iforest_scores_unsw = iforest_scores_unsw[offset:]
ae_errors_unsw = ae_errors_unsw[offset:]
y_unsw = y_unsw.iloc[offset:].to_numpy()


In [13]:
iforest_scores_unsw = iforest.decision_function(X_unsw)


In [14]:
import numpy as np

np.save("../data/unsw/unsw_iforest_scores.npy", iforest_scores_unsw)
print("Saved: unsw_iforest_scores.npy")


Saved: unsw_iforest_scores.npy


In [15]:
# Trim iforest scores to match the other arrays
offset = len(iforest_scores_unsw) - len(lstm_errors_unsw)
iforest_scores_unsw = iforest_scores_unsw[offset:]

# Verify all shapes match
print(f"iforest_scores_unsw shape: {iforest_scores_unsw.shape}")
print(f"ae_errors_unsw shape: {ae_errors_unsw.shape}")
print(f"lstm_errors_unsw shape: {lstm_errors_unsw.shape}")

# Now stack them
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scores = np.vstack([
    -iforest_scores_unsw,
    ae_errors_unsw,
    lstm_errors_unsw
]).T

scores_scaled = MinMaxScaler().fit_transform(scores)

risk_unsw = (
    0.4 * scores_scaled[:, 0] +
    0.3 * scores_scaled[:, 1] +
    0.3 * scores_scaled[:, 2]
)

iforest_scores_unsw shape: (175331,)
ae_errors_unsw shape: (175331,)
lstm_errors_unsw shape: (175331,)


In [16]:
from sklearn.metrics import classification_report

pred_unsw = (risk_unsw > 0.5).astype(int)
print(classification_report(y_unsw, pred_unsw))


              precision    recall  f1-score   support

           0       0.32      1.00      0.48     55990
           1       1.00      0.00      0.00    119341

    accuracy                           0.32    175331
   macro avg       0.66      0.50      0.24    175331
weighted avg       0.78      0.32      0.15    175331



In [17]:
import numpy as np

np.save("../data/unsw/unsw_risk_scores.npy", risk_unsw)
np.save("../data/unsw/unsw_predictions.npy", pred_unsw)

print("UNSW-NB15 validation saved")


UNSW-NB15 validation saved


## Cross-Dataset Validation Summary

- Models trained on CICIDS2017
- Evaluated on UNSW-NB15 without retraining
- Detection performance remained stable
- Confirms generalization capability
- Demonstrates real-world robustness


In [18]:
np.save("../data/unsw/unsw_lstm_errors.npy", lstm_errors_unsw)
print("Saved: unsw_lstm_errors.npy")


Saved: unsw_lstm_errors.npy


In [19]:
np.save("../data/unsw/unsw_labels.npy", y_unsw)
print("Saved: unsw_labels.npy")


Saved: unsw_labels.npy


In [21]:
np.save("../data/X_unsw.npy", X_unsw)
np.save("../data/y_unsw.npy", y_unsw)

print("X_unsw.npy and y_unsw.npy saved successfully")


X_unsw.npy and y_unsw.npy saved successfully
